# หลักสูตรวิทยาศาสตร์ข้อมูลสำหรับผู้บริหาร: พื้นฐานและการเตรียมความพร้อม
## ตอนที่ 3: การจัดการและวิเคราะห์ตารางข้อมูลด้วย Pandas โดยเปรียบเทียบกับ Microsoft Excel
### กรณีศึกษา: ชุดข้อมูลยอดขายวิดีโอเกมทั่วโลก (Video Game Sales Dataset ผ่าน KaggleHub)

---

### วัตถุประสงค์การเรียนรู้
1. ทำความเข้าใจการเปลี่ยนผ่านจากการทำงานบน Microsoft Excel สู่การประมวลผลอัตโนมัติด้วย Pandas DataFrame
2. สามารถอ่านและเปรียบเทียบคำสั่งใน Pandas กับเครื่องมือที่คุ้นเคยใน Excel (เช่น AutoFilter, VLOOKUP, Pivot Table)
3. เข้าใจกระบวนการตรวจสอบคุณภาพข้อมูลเบื้องต้น (Data Quality Audit) ด้วยคำสั่งเชิงสถิติและโครงสร้าง บนชุดข้อมูลจริงระดับ 16,000+ แถว
4. เข้าใจตรรกะการทำความสะอาดข้อมูลสูญหาย (Missing Values) และการรวมตารางข้อมูลจากหลายแหล่ง (Data Merging)

## 1. บทนำ: ทำไมองค์กรจึงย้ายการวิเคราะห์จาก Excel มาสู่ Pandas

Microsoft Excel เป็นเครื่องมือที่ยอดเยี่ยมสำหรับการสร้างแบบจำลองการเงินและการจัดการข้อมูลทั่วไป อย่างไรก็ตาม เมื่อองค์กรขยายตัวและมีปริมาณข้อมูลเพิ่มขึ้น ข้อจำกัดของ Excel จะเริ่มส่งผลกระทบต่อประสิทธิภาพการดำเนินงาน:

| มิติการเปรียบเทียบ | Microsoft Excel | Pandas (Python) |
|---|---|---|
| **ขีดจำกัดขนาดข้อมูล** | 1,048,576 แถว (เริ่มตอบสนองช้าเมื่อเกิน 200,000 แถว) | ไม่จำกัด ขึ้นอยู่กับขนาด RAM ของระบบ (รองรับหลายล้านแถวได้อย่างรวดเร็ว) |
| **การทำงานอัตโนมัติ (Automation)** | ต้องบันทึก Macro หรือทำซ้ำด้วยมือทุกสัปดาห์/เดือน | เขียนคำสั่งครั้งเดียว สามารถรันซ้ำตามรอบเวลาได้อัตโนมัติ (Pipeline) |
| **การตรวจสอบย้อนกลับ (Auditability)** | สูตรคำนวณอาจถูกเขียนทับในบางเซลล์โดยไม่รู้ตัว (Human Error) | โค้ดถูกบันทึกเป็นขั้นตอนเชิงตรรกะ ข้อมูลต้นฉบับไม่ถูกแตะต้อง (Non-destructive) |
| **การเชื่อมต่อแหล่งข้อมูล** | การเชื่อมโยงไฟล์ข้ามเครื่องมักเกิดปัญหาลิงก์ขาด | เชื่อมต่อโดยตรงกับฐานข้อมูล SQL, Cloud Storage, และ API ได้อย่างมั่นคง |

## 2. ตารางเทียบเคียงฟังก์ชันการทำงาน: Excel เทียบกับ Pandas

| การทำงานใน Excel | เครื่องมือ / ฟังก์ชันใน Excel | คำสั่งใน Pandas |
|---|---|---|
| **เปิดไฟล์ข้อมูล** | File > Open หรือ Data Query | `pd.read_csv()` หรือ `pd.read_excel()` |
| **ดูขนาดตาราง** | ดูแถวและคอลัมน์สุดท้าย | `df.shape` (ส่งกลับจำนวนแถวและจำนวนคอลัมน์) |
| **ดูประเภทข้อมูลและช่องว่าง** | ดู Format Cells และคอลัมน์ | `df.info()` |
| **สรุปสถิติตัวเลข** | AVERAGE, STDEV, MIN, MAX | `df.describe()` |
| **การกรองข้อมูล** | ตัวกรองอัตโนมัติ (AutoFilter) | Boolean Indexing เช่น `df[df['Global_Sales'] > 10.0]` |
| **การจัดการค่าว่าง** | ค้นหา Blank Cells แล้วลบหรือเติม | `df.dropna()` หรือ `df.fillna()` |
| **การสรุปกลุ่มข้อมูล** | Pivot Table | `df.groupby('Genre')['Global_Sales'].agg(['sum', 'mean'])` |
| **การดึงข้อมูลข้ามตาราง** | VLOOKUP หรือ XLOOKUP | `pd.merge(table_a, table_b, on='Key', how='left')` |

## 3. การนำเข้าชุดข้อมูลสากลผ่าน KaggleHub และการตรวจสอบคุณภาพข้อมูล

เราจะดาวน์โหลดชุดข้อมูลจริง `gregorut/videogamesales` ผ่านไลบรารี `kagglehub` แล้วนำเข้าไฟล์ `vgsales.csv` เข้าสู่ DataFrame

คำอธิบายคอลัมน์ในชุดข้อมูล:
- `Rank`: อันดับยอดขายรวมทั่วโลก
- `Name`: ชื่อเกม
- `Platform`: แพลตฟอร์มเครื่องเล่นเกม (เช่น PS4, X360, Wii, PC)
- `Year`: ปีที่วางจำหน่าย
- `Genre`: ประเภทของเกม (เช่น Action, Sports, Shooter)
- `Publisher`: บริษัทผู้จัดจำหน่าย
- `NA_Sales`: ยอดขายในทวีปอเมริกาเหนือ (หน่วย: ล้านชุด)
- `EU_Sales`: ยอดขายในทวีปยุโรป (หน่วย: ล้านชุด)
- `JP_Sales`: ยอดขายในประเทศญี่ปุ่น (หน่วย: ล้านชุด)
- `Other_Sales`: ยอดขายในภูมิภาคอื่นๆ ทั่วโลก (หน่วย: ล้านชุด)
- `Global_Sales`: ยอดขายรวมทั่วโลก (หน่วย: ล้านชุด)

In [ ]:
# ติดตั้ง kagglehub สำหรับดาวน์โหลดชุดข้อมูล
%pip install -q kagglehub

# นำเข้าไลบรารี kagglehub และ pandas
import kagglehub
import pandas as pd

# ดาวน์โหลดชุดข้อมูล Video Game Sales จาก Kaggle
path = kagglehub.dataset_download("gregorut/videogamesales")
print("ตำแหน่งไฟล์ข้อมูลในเครื่องเสมือน:", path)

# นำเข้าไฟล์ vgsales.csv เข้าสู่ตัวแปรตารางข้อมูล df (DataFrame)
df = pd.read_csv(path + "/vgsales.csv")

# แสดงผล 5 แถวแรกของตารางข้อมูลด้วย display()
display(df.head())

### การตรวจสอบขนาดและโครงสร้างข้อมูลด้วย `.shape` และ `.info()`
ใน Excel การเปิดไฟล์ที่มีจำนวนแถวเกินหมื่นแถวอาจเริ่มตอบสนองช้า ใน Pandas เราสามารถใช้คำสั่ง `.shape` เพื่อดูมิติข้อมูล และ `.info()` เพื่อตรวจสอบว่ามีคอลัมน์ใดที่มีค่าสูญหาย (Missing Data) หรือไม่

In [ ]:
# ตรวจสอบจำนวนแถวและจำนวนคอลัมน์
print("ขนาดของตารางข้อมูล (จำนวนแถว, จำนวนคอลัมน์):", df.shape)

print("-" * 80)
# แสดงรายละเอียดของแต่ละคอลัมน์ ชนิดข้อมูล และจำนวนข้อมูลที่ไม่เป็นค่าว่าง
df.info()

### การสรุปสถิติตัวเลขภาพรวมด้วย `.describe()`
คำสั่งนี้เทียบเท่ากับการเขียนสูตรคำนวณหาค่าเฉลี่ย (Mean) ส่วนเบี่ยงเบนมาตรฐาน (Std) ค่าต่ำสุด (Min) ควอร์ไทล์ และค่าสูงสุด (Max) ของทุกคอลัมน์ตัวเลขพร้อมกันในเสี้ยววินาที

In [ ]:
# สรุปสถิติเชิงพรรณนาสำหรับคอลัมน์ยอดขายและปีที่เปิดตัว
df.describe()

## 4. การกรองข้อมูลตามเงื่อนไขทางธุรกิจ (Data Filtering: AutoFilter เทียบกับ Boolean Indexing)

ใน Excel การกรองทำได้โดยคลิกเปิด AutoFilter แล้วติ๊กเลือกค่าที่ต้องการ ซึ่งเสี่ยงต่อการที่พนักงานลืมปลดฟิลเตอร์หรือเลือกไม่ครบ
ใน Pandas จะใช้การเขียนเงื่อนไขทางตรรกะในวงเล็บก้ามปู ซึ่งมีความชัดเจน ตรวจสอบย้อนหลังได้ และรันซ้ำอัตโนมัติ

In [ ]:
# ตัวอย่างที่ 1: กรองเฉพาะเกมบนแพลตฟอร์ม PlayStation 4 ('PS4')
ps4_games = df[df["Platform"] == "PS4"]
print("จำนวนเกมบนแพลตฟอร์ม PS4:", len(ps4_games))
display(ps4_games.head(3))

# ตัวอย่างที่ 2: กรองแบบหลายเงื่อนไข (เกมแนว Action ที่มียอดขายทั่วโลก Global_Sales ตั้งแต่ 5 ล้านชุดขึ้นไป)
top_action_games = df[(df["Genre"] == "Action") & (df["Global_Sales"] >= 5.0)]
print("จำนวนเกม Action ระดับยอดขายมากกว่า 5 ล้านชุด:", len(top_action_games))
display(top_action_games[["Name", "Platform", "Year", "Publisher", "Global_Sales"]].head(5))

## 5. การจัดการข้อมูลสูญหาย (Handling Missing Values)

เมื่อตรวจสอบด้วย `df.isna().sum()` จะพบว่าในชุดข้อมูลจริงนี้มีคอลัมน์ที่มีข้อมูลขาดหาย:
- `Year`: มีข้อมูลสูญหายประมาณ 271 รายการ
- `Publisher`: มีข้อมูลสูญหายประมาณ 58 รายการ

แนวทางจัดการทางธุรกิจ:
1. สำหรับ `Publisher` ที่ไม่ระบุ: หากเราต้องการความแม่นยำของรายงานบริษัทผู้จัดจำหน่าย เราสามารถใช้ `dropna(subset=["Publisher"])` เพื่อตัดรายการที่ไม่ระบุออก
2. สำหรับ `Year` ที่ขาดหาย: สามารถแทนที่ด้วยค่ามัธยฐาน (Median Year) หรือใช้ `fillna()` เพื่อไม่ให้สูญเสียยอดขายรวมของแถวนั้น

In [ ]:
# ตรวจสอบจำนวนข้อมูลสูญหายในแต่ละคอลัมน์
print("จำนวนค่าว่างในแต่ละคอลัมน์:")
print(df.isna().sum())

print("-" * 80)

# ทำสำเนาข้อมูลเพื่อเข้าสู่กระบวนการทำความสะอาด (Non-destructive approach)
df_cleaned = df.copy()

# 1. ตัดแถวที่ไม่มีข้อมูล Publisher ออกเพื่อความถูกต้องของรายงานบริษัทคู่ค้า
df_cleaned = df_cleaned.dropna(subset=["Publisher"])

# 2. เติมเต็มค่าว่างในคอลัมน์ Year ด้วยค่ามัธยฐานของปีที่วางจำหน่าย
median_year = df_cleaned["Year"].median()
df_cleaned["Year"] = df_cleaned["Year"].fillna(median_year)

# ตรวจสอบผลลัพธ์หลังทำความสะอาด
print("จำนวนค่าว่างหลังการทำความสะอาด:")
print(df_cleaned.isna().sum())

## 6. การสรุปผลเชิงกลุ่ม (GroupBy: Pivot Table ใน Excel)

คำสั่ง `.groupby()` ใน Pandas ทำงานเทียบเท่ากับการลากฟิลด์ลงใน Pivot Table ของ Excel:
- ดึงคอลัมน์หมวดหมู่ (เช่น `Genre`) มาไว้ที่ Rows
- นำคอลัมน์ตัวเลข (เช่น `Global_Sales`, `NA_Sales`, `EU_Sales`, `JP_Sales`) มาคำนวณผลรวม (Sum) หรือค่าเฉลี่ย (Mean) ในส่วน Values

In [ ]:
# การทำ Pivot Table สรุปยอดขายรวมและค่าเฉลี่ย แยกตามประเภทเกม (Genre)
genre_pivot = df_cleaned.groupby("Genre").agg({
    "Global_Sales": ["sum", "mean"],
    "NA_Sales": "sum",
    "EU_Sales": "sum",
    "JP_Sales": "sum",
    "Name": "count"
})

# ปรับแต่งชื่อคอลัมน์ให้อ่านเข้าใจง่ายสำหรับรายงานฝ่ายบริหาร
genre_pivot.columns = [
    "ยอดขายรวมทั่วโลก (ล้านชุด)",
    "ยอดขายเฉลี่ยต่อเกม (ล้านชุด)",
    "ยอดขายรวมอเมริกาเหนือ (ล้านชุด)",
    "ยอดขายรวมยุโรป (ล้านชุด)",
    "ยอดขายรวมญี่ปุ่น (ล้านชุด)",
    "จำนวนเกมทั้งหมด"
]

# เรียงลำดับจากประเภทเกมที่สร้างยอดขายทั่วโลกสูงสุดลงมา
genre_pivot = genre_pivot.sort_values(by="ยอดขายรวมทั่วโลก (ล้านชุด)", ascending=False)
display(genre_pivot)

## 7. การรวมตารางข้อมูล (Data Merging: VLOOKUP / XLOOKUP ใน Excel)

ในงานธุรกิจจริง ข้อมูลมักแยกจัดเก็บในตารางหลัก (Master Table) ตัวอย่างเช่น เรามีตารางระบุประเทศต้นกำเนิดและทวีปของบริษัทผู้จัดจำหน่าย (Publisher Headquarter Master)

ใน Excel ผู้ใช้งานต้องเขียนสูตร `=VLOOKUP(F2, Master!A:B, 2, FALSE)` ซึ่งทำงานช้าเมื่อข้อมูลมี 16,000+ แถว
ใน Pandas เราใช้คำสั่ง `pd.merge()` เพื่อผสานข้อมูลได้อย่างรวดเร็วและแม่นยำ

In [ ]:
# สร้างตารางข้อมูลมาสเตอร์ของบริษัทผู้จัดจำหน่ายเกมชั้นนำ
publisher_master_data = pd.DataFrame({
    "Publisher": [
        "Nintendo", 
        "Electronic Arts", 
        "Activision", 
        "Sony Computer Entertainment", 
        "Ubisoft", 
        "Take-Two Interactive"
    ],
    "HQ_Country": ["Japan", "USA", "USA", "Japan", "France", "USA"],
    "Market_Tier": ["Tier 1", "Tier 1", "Tier 1", "Tier 1", "Tier 1", "Tier 1"]
})

print("ตารางมาสเตอร์บริษัทผู้จัดจำหน่าย:")
display(publisher_master_data)

print("-" * 80)

# ทำการ Left Merge เพื่อดึงข้อมูล HQ_Country เข้าสู่ตารางยอดขายหลัก (เทียบเท่า VLOOKUP)
df_merged = pd.merge(
    left=df_cleaned,
    right=publisher_master_data,
    on="Publisher",
    how="left"
)

# แสดงผลตัวอย่าง 5 แถวแรกที่ผ่านการจับคู่ข้อมูล
display(df_merged[["Rank", "Name", "Platform", "Publisher", "HQ_Country", "Global_Sales"]].head(5))

## 8. โค้ดตัวอย่างแบบบูรณาการ: สรุปผลงานของบริษัทผู้ผลิตรายใหญ่

ตัวอย่างกระบวนการประมวลผลตั้งแต่ต้นจนจบ (End-to-End Pipeline) เพื่อสรุปยอดขายรวมและส่วนแบ่งการตลาดของบริษัทระดับ Tier 1

In [ ]:
# กรองเฉพาะเกมที่ผลิตโดยบริษัทระดับ Tier 1
tier1_games = df_merged[df_merged["Market_Tier"] == "Tier 1"]

# สรุปยอดขายตามประเทศสำนักงานใหญ่ (HQ_Country) และบริษัท (Publisher)
tier1_summary = tier1_games.groupby(["HQ_Country", "Publisher"]).agg({
    "Global_Sales": "sum",
    "NA_Sales": "sum",
    "JP_Sales": "sum",
    "Name": "count"
}).reset_index()

# ปรับชื่อคอลัมน์สำหรับรายงาน
tier1_summary.columns = [
    "ประเทศต้นกำเนิด", 
    "บริษัทผู้จัดจำหน่าย", 
    "ยอดขายรวมทั่วโลก (ล้านชุด)", 
    "ยอดขายอเมริกาเหนือ (ล้านชุด)", 
    "ยอดขายญี่ปุ่น (ล้านชุด)", 
    "จำนวนเกมที่เปิดตัว"
]

# จัดเรียงตามยอดขายรวมทั่วโลกจากมากไปน้อย
tier1_summary = tier1_summary.sort_values(by="ยอดขายรวมทั่วโลก (ล้านชุด)", ascending=False)
display(tier1_summary)

## 9. ส่วนฝึกปฏิบัติสำหรับผู้เรียน (Hands-on Exercise)

### โจทย์ธุรกิจสำหรับการประเมินผล
ในฐานะผู้จัดการฝ่ายกลยุทธ์ผลิตภัณฑ์ ท่านได้รับมอบหมายให้ศึกษาตลาดเกมบนเครื่อง **PlayStation 4 (`Platform == 'PS4'`)**:
1. **โจทย์ที่ 1 (การกรองข้อมูล):** กรองข้อมูลจากตาราง `df_cleaned` โดยเลือกเฉพาะเกมบนแพลตฟอร์ม **`PS4`** ที่มียอดขายทั่วโลก **`Global_Sales >= 2.0`** (ล้านชุด) จัดเก็บในตัวแปร `top_ps4_games`
2. **โจทย์ที่ 2 (การคำนวณทางสถิติ):** จากตัวแปร `top_ps4_games` จงคำนวณ:
   - ยอดขายรวมทั่วโลกของกลุ่มนี้ (`total_global_sales`)
   - ค่าเฉลี่ยยอดขายในทวีปอเมริกาเหนือต่อเกม (`avg_na_sales`)
3. **โจทย์ที่ 3 (Pivot Table ด้วย GroupBy):** ใช้คำสั่ง `.groupby()` สรุปผลยอดขายรวมทั่วโลก (`Global_Sales`) แยกตามประเภทเกม (**`Genre`**) สำหรับกลุ่ม `top_ps4_games` พร้อมจัดเรียงจากประเภทที่ขายดีที่สุด

In [ ]:
# TODO: เขียนโค้ดตอบโจทย์ที่ 1, 2, และ 3 ด้านล่างนี้
# -----------------------------------------------------------------------------
# 1. กรองเกมบนแพลตฟอร์ม PS4 ที่มียอดขาย Global_Sales >= 2.0 ล้านชุด


# 2. คำนวณยอดขายรวมทั่วโลก และค่าเฉลี่ยยอดขายในอเมริกาเหนือ


# 3. สรุปยอดขายรวม Global_Sales แยกตาม Genre โดยใช้ groupby


# -----------------------------------------------------------------------------

---
## เฉลยและคำอธิบายแนวคิด (Solution & Managerial Concept)

ส่วนนี้จัดทำขึ้นเพื่อให้ผู้บริหารสามารถตรวจทานและทำความเข้าใจแนวทางการเขียนคำสั่งที่ถูกต้อง

In [ ]:
# เฉลยโจทย์ที่ 1: การกรองข้อมูลแบบ 2 เงื่อนไขด้วยเครื่องหมาย &
top_ps4_games = df_cleaned[(df_cleaned["Platform"] == "PS4") & (df_cleaned["Global_Sales"] >= 2.0)]
print("จำนวนเกม PS4 ที่มียอดขายตั้งแต่ 2 ล้านชุดขึ้นไป:", len(top_ps4_games))
display(top_ps4_games[["Rank", "Name", "Genre", "Publisher", "Global_Sales"]].head(5))

# เฉลยโจทย์ที่ 2: การคำนวณทางสถิติ
total_global_sales = top_ps4_games["Global_Sales"].sum()
avg_na_sales = top_ps4_games["NA_Sales"].mean()
print(f"ยอดขายรวมทั่วโลกของเกมกลุ่มนี้: {total_global_sales:,.2f} ล้านชุด")
print(f"ยอดขายเฉลี่ยในอเมริกาเหนือต่อเกม: {avg_na_sales:,.2f} ล้านชุด")

# เฉลยโจทย์ที่ 3: สรุปผลด้วย groupby แยกตาม Genre พร้อมเรียงลำดับ
ps4_genre_summary = top_ps4_games.groupby("Genre")["Global_Sales"].sum().reset_index()
ps4_genre_summary.columns = ["ประเภทเกม", "ยอดขายรวมทั่วโลก (ล้านชุด)"]
ps4_genre_summary = ps4_genre_summary.sort_values(by="ยอดขายรวมทั่วโลก (ล้านชุด)", ascending=False)
display(ps4_genre_summary)

### คำอธิบายแนวคิดการเลือกใช้คำสั่งสำหรับผู้บริหาร
1. **การกรองข้อมูลความจุสูง:** การใช้คำสั่งเดียวกรองข้อมูล 16,500+ แถว ช่วยประหยัดเวลาและลดความผิดพลาดของมนุษย์เมื่อเทียบกับการใช้เมาส์คลิกฟิลเตอร์ใน Excel
2. **การคำนวณที่แม่นยำ:** การดึงคอลัมน์ตัวเลขมาหา `.sum()` และ `.mean()` โดยตรง ป้องกันปัญหาช่วงเซลล์หลุด (เช่น ลืมรวมแถวที่เพิ่มเข้ามาใหม่ในสูตร `=SUM(A1:A500)`)
3. **ความพร้อมในการส่งต่อผลลัพธ์:** ตารางที่ได้จาก `.reset_index()` อยู่ในรูปแบบโครงสร้างตารางสัมพันธ์ (Relational Table) พร้อมนำไปสร้าง Dashboard หรือเชื่อมต่อกับระบบ Business Intelligence (BI) ขององค์กรได้ทันที